# **Knowledge Base Builder+RAG**

## **Baseline RAG**

In [ ]:
# ================================================================
# FULL RAG PIPELINE
# KB Construction → Chunking → Embedding → Retrieval → LLM → Eval
# ================================================================

# ── Install ───────────────────────────────────────────────────
!pip install faiss-cpu sentence-transformers anthropic -q

import json, os, hashlib, time
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from collections import defaultdict, Counter
import anthropic

os.makedirs("/content/rag_pipeline", exist_ok=True)


# ================================================================
# STEP 1 — LOAD UNIFIED DATASET
# ================================================================

with open("/content/unified_dataset/unified_train.json") as f:
    all_samples = json.load(f)

print(f"Total samples: {len(all_samples)}")


# ================================================================
# STEP 2 — SOURCE-AWARE CHUNKING + KB CONSTRUCTION
# ================================================================

CHUNK_WORD_LIMIT = 350   # max words per chunk
CHUNK_OVERLAP    = 50    # overlap in words

def hash_text(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def get_ctx_text(sample):
    docs = sample["context"]["documents"]
    return " ".join(d["text"].strip() for d in docs if d.get("text","").strip())

def chunk_text(text, max_words=CHUNK_WORD_LIMIT, overlap=CHUNK_OVERLAP):
    """Split long text into overlapping word-level chunks."""
    words  = text.split()
    if len(words) <= max_words:
        return [text]
    chunks = []
    start  = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return chunks

def needs_chunking(text, source):
    """Only chunk sources with long contexts."""
    word_count = len(text.split())
    if source == "contract_nli" and word_count > 300:
        return True
    if source == "quac" and word_count > 400:
        return True
    return False   # hotpotqa and sharc are short — no chunking


# Build KB with chunking
kb_entries   = []
chunk_id_ctr = 0
seen_chunks  = {}   # chunk_hash → kb index

# Group samples by context hash first
ctx_groups = defaultdict(list)
for s in all_samples:
    ctx_text = get_ctx_text(s)
    if not ctx_text.strip():
        continue
    ctx_groups[hash_text(ctx_text)].append(s)

print(f"Unique contexts before chunking: {len(ctx_groups)}")

for ctx_hash, samples in ctx_groups.items():
    ctx_text   = get_ctx_text(samples[0])
    source     = samples[0]["metadata"]["source"]
    docs       = samples[0]["context"]["documents"]

    # Source-aware chunking
    if needs_chunking(ctx_text, source):
        chunks = chunk_text(ctx_text)
    else:
        chunks = [ctx_text]

    # Shared linkage info across all chunks of this context
    linked_ids     = [s["id"]     for s in samples]
    linked_actions = [s["action"] for s in samples]
    linked_queries = [s["query"]  for s in samples]

    action_dist = dict(Counter(linked_actions))

    for chunk_idx, chunk in enumerate(chunks):
        chunk_hash = hash_text(chunk)
        if chunk_hash in seen_chunks:
            continue   # skip duplicate chunks

        # Doc metadata
        doc_meta = {}
        if source == "quac":
            doc_meta = {"dialogue_id": samples[0]["metadata"].get("dialogue_id","")}
        elif source == "sharc":
            doc_meta = {"url": docs[0].get("url","") if docs else ""}
        elif source == "contract_nli":
            doc_meta = {
                "doc_id"   : docs[0].get("doc_id","") if docs else "",
                "file_name": docs[0].get("file_name","") if docs else "",
                "chunk_idx": chunk_idx,
                "total_chunks": len(chunks)
            }
        elif source == "hotpotqa":
            doc_meta = {"supporting_titles": [d.get("doc_id","") for d in docs]}

        entry = {
            "kb_id"              : f"kb_{chunk_id_ctr:07d}",
            "chunk_hash"         : chunk_hash,
            "parent_ctx_hash"    : ctx_hash,
            "chunk_idx"          : chunk_idx,
            "total_chunks"       : len(chunks),
            "source"             : source,
            "text"               : chunk,
            "word_count"         : len(chunk.split()),
            "doc_meta"           : doc_meta,
            "linked_sample_ids"  : linked_ids,
            "linked_actions"     : linked_actions,
            "linked_queries"     : linked_queries,
            "action_distribution": action_dist,
            "num_linked"         : len(linked_ids),
        }

        seen_chunks[chunk_hash] = chunk_id_ctr
        kb_entries.append(entry)
        chunk_id_ctr += 1

print(f"Total KB chunks after source-aware chunking: {len(kb_entries)}")

# Stats
src_counts = Counter(e["source"] for e in kb_entries)
print("\n  Chunks per source:")
for src, cnt in src_counts.items():
    avg_words = np.mean([e["word_count"] for e in kb_entries if e["source"]==src])
    print(f"  {src:15} : {cnt:6} chunks | avg {avg_words:.0f} words/chunk")

# Save KB
kb_path = "/content/rag_pipeline/knowledge_base.json"
with open(kb_path, "w") as f:
    json.dump(kb_entries, f, indent=2)
print(f"\nSaved KB → {kb_path}")


In [ ]:


# ================================================================
# STEP 3 — EMBED + FAISS INDEX
# ================================================================

print("\nLoading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [e["text"] for e in kb_entries]
print(f"Embedding {len(texts)} chunks...")

embeddings = embedder.encode(
    texts,
    batch_size     = 128,
    show_progress_bar  = True,
    convert_to_numpy   = True,
    normalize_embeddings = True
)

print(f"Embedding shape: {embeddings.shape}")

# FAISS index — exact cosine via normalized inner product
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index: {index.ntotal} vectors | dim={dim}")

# Save
np.save("/content/rag_pipeline/kb_embeddings.npy", embeddings)
faiss.write_index(index, "/content/rag_pipeline/kb_faiss.index")

# kb_id → array position map
kb_id_map = {e["kb_id"]: i for i, e in enumerate(kb_entries)}
with open("/content/rag_pipeline/kb_id_map.json","w") as f:
    json.dump(kb_id_map, f)

print("Saved embeddings + FAISS index + id map")


# ================================================================
# STEP 4 — RETRIEVAL MODULE
# ================================================================

def retrieve(query, top_k=5, source_filter=None):
    """
    Retrieve top-k chunks for a query.
    Optional source_filter: list of sources to restrict to e.g. ["quac","sharc"]
    """
    q_emb = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    # Search more than top_k if filtering
    search_k = top_k * 5 if source_filter else top_k
    scores, indices = index.search(q_emb, search_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(kb_entries):
            continue
        entry = kb_entries[idx]
        if source_filter and entry["source"] not in source_filter:
            continue
        results.append({
            "kb_id"          : entry["kb_id"],
            "source"         : entry["source"],
            "score"          : round(float(score), 4),
            "text"           : entry["text"],
            "action_dist"    : entry["action_distribution"],
            "linked_queries" : entry["linked_queries"][:3],
            "word_count"     : entry["word_count"],
        })
        if len(results) == top_k:
            break

    return results


# ================================================================
# STEP 5 — PROMPT-BASED DECISION LAYER (Mistral 7B Instruct)
# ================================================================

!pip install transformers accelerate bitsandbytes -q

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"   # ungated

# 4-bit quantization — fits in Colab T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("Loading Mistral 7B Instruct (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True
)
model.eval()
print("Model loaded.")


SYSTEM_PROMPT = """You are a decision-aware intelligent assistant.

Given a user query and retrieved context passages, you must:

1. Carefully read the context.
2. Decide which action is appropriate:
   - ANSWER   → if the context contains sufficient information to answer the query
   - ASK      → if key information is missing and a clarification question would help
   - ABSTAIN  → if the query cannot be answered from the context at all

3. Output your response in this EXACT format:
ACTION: <ANSWER|ASK|ABSTAIN>
RESPONSE: <your answer, clarification question, or abstain statement>

Rules:
- Never guess or hallucinate. If uncertain → ASK or ABSTAIN.
- If ASKing, ask ONE focused clarification question.
- If ABSTAINing, say exactly why you cannot answer.
- Be concise."""


def build_prompt(query, retrieved_chunks, history=None):
    ctx_block = ""
    for i, chunk in enumerate(retrieved_chunks, 1):
        ctx_block += (f"\n[Context {i} | source={chunk['source']} "
                      f"| score={chunk['score']}]\n")
        ctx_block += chunk["text"][:800] + "\n"

    history_block = ""
    if history:
        history_block = "Conversation so far:\n"
        for turn in history:
            history_block += f"  User: {turn['query']}\n"
            history_block += f"  Assistant [{turn['action']}]: {turn['response']}\n"
        history_block += "\n"

    user_content = f"""{history_block}Query:
{query}

Retrieved Context:
{ctx_block}
"""
    # Mistral instruct chat template
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_content}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize        = False,
        add_generation_prompt = True
    )


def rag_infer(query, top_k=5, source_filter=None, history=None):
    """Full RAG inference: retrieve → prompt → Mistral → parse."""
    chunks     = retrieve(query, top_k=top_k, source_filter=source_filter)
    prompt_str = build_prompt(query, chunks, history)

    inputs = tokenizer(
        prompt_str,
        return_tensors = "pt",
        truncation     = True,
        max_length     = 3072
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens  = 200,
            do_sample       = False,   # greedy — deterministic for eval
            temperature     = 1.0,
            pad_token_id    = tokenizer.eos_token_id,
            eos_token_id    = tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Parse structured output — same logic as before
    action   = "ABSTAIN"   # default safe
    resp_txt = raw_output

    for line in raw_output.split("\n"):
        line_strip = line.strip()
        if line_strip.startswith("ACTION:"):
            action_raw = line_strip.replace("ACTION:", "").strip()
            if action_raw in ["ANSWER", "ASK", "ABSTAIN"]:
                action = action_raw
        if line_strip.startswith("RESPONSE:"):
            resp_txt = line_strip.replace("RESPONSE:", "").strip()

    return {
        "query"           : query,
        "action"          : action,
        "response"        : resp_txt,
        "raw_output"      : raw_output,
        "retrieved_chunks": chunks,
        "num_retrieved"   : len(chunks),
    }


# ================================================================
# STEP 6 — EVALUATION ON HELD-OUT SAMPLE
# ================================================================
# We sample balanced set from unified dataset for eval
# Ground truth = action label already in unified dataset

import random
random.seed(42)

def sample_eval_set(all_samples, n_per_action=100):
    """Sample balanced eval set — n per ANSWER/ASK/ABSTAIN."""
    by_action = defaultdict(list)
    for s in all_samples:
        by_action[s["action"]].append(s)
    eval_set = []
    for action, samples in by_action.items():
        eval_set.extend(random.sample(samples, min(n_per_action, len(samples))))
    random.shuffle(eval_set)
    return eval_set

eval_samples = sample_eval_set(all_samples, n_per_action=100)
print(f"Eval set size: {len(eval_samples)}")
print(f"Distribution: {Counter(s['action'] for s in eval_samples)}")


def evaluate_rag(eval_samples, top_k=5, sleep_sec=0.3):
    """
    Run RAG on eval set and compute:
    - Decision accuracy (predicted action vs ground truth action)
    - Hallucination rate (predicted ANSWER when GT is ASK/ABSTAIN)
    - Over-answer rate
    - Per-action breakdown
    """
    results   = []
    correct   = 0
    hallucin  = 0   # predicted ANSWER when GT != ANSWER
    over_ans  = 0   # predicted ANSWER when GT is ASK or ABSTAIN

    pred_actions = []
    gt_actions   = []

    for i, sample in enumerate(eval_samples):
        if i % 20 == 0:
            print(f"  Evaluating {i+1}/{len(eval_samples)}...")

        try:
            out = rag_infer(
                query          = sample["query"],
                top_k          = top_k,
                source_filter  = None
            )
        except Exception as ex:
            print(f"  Error on sample {i}: {ex}")
            out = {"action": "ABSTAIN", "response": "error",
                   "retrieved_chunks": [], "raw_output": ""}

        gt_action   = sample["action"]
        pred_action = out["action"]

        pred_actions.append(pred_action)
        gt_actions.append(gt_action)

        is_correct = (pred_action == gt_action)
        if is_correct:
            correct += 1

        # Hallucination: model answers when it shouldn't
        if pred_action == "ANSWER" and gt_action != "ANSWER":
            hallucin += 1
            over_ans += 1

        results.append({
            "sample_id"    : sample["id"],
            "source"       : sample["metadata"]["source"],
            "query"        : sample["query"],
            "gt_action"    : gt_action,
            "pred_action"  : pred_action,
            "response"     : out["response"],
            "correct"      : is_correct,
            "top_retrieved": [
                {"source": c["source"], "score": c["score"]}
                for c in out["retrieved_chunks"]
            ],
        })

        # time.sleep(sleep_sec)   # rate limit buffer

    # ── Metrics ──────────────────────────────────────────────
    n = len(results)

    print("\n" + "="*55)
    print("  RAG EVALUATION RESULTS")
    print("="*55)
    print(f"  Total evaluated         : {n}")
    print(f"  Decision Accuracy       : {correct/n*100:.1f}%  ({correct}/{n})")
    print(f"  Hallucination Rate      : {hallucin/n*100:.1f}%  "
          f"(answered when should ASK/ABSTAIN)")
    print(f"  Over-answer Rate        : {over_ans/n*100:.1f}%")

    print(f"\n  --- Per-action accuracy ---")
    for act in ["ANSWER","ASK","ABSTAIN"]:
        act_results = [r for r in results if r["gt_action"] == act]
        if not act_results:
            continue
        act_correct = sum(r["correct"] for r in act_results)
        print(f"  {act:10} : {act_correct}/{len(act_results)}  "
              f"({100*act_correct/len(act_results):.1f}%)")

    print(f"\n  --- Confusion (GT → Predicted) ---")
    confusion = defaultdict(Counter)
    for r in results:
        confusion[r["gt_action"]][r["pred_action"]] += 1
    for gt, preds in sorted(confusion.items()):
        print(f"  GT={gt:10} → {dict(preds)}")

    print(f"\n  --- Accuracy by source ---")
    src_res = defaultdict(list)
    for r in results:
        src_res[r["source"]].append(r["correct"])
    for src, vals in sorted(src_res.items()):
        print(f"  {src:15} : {sum(vals)}/{len(vals)}  "
              f"({100*sum(vals)/len(vals):.1f}%)")

    # Save results
    out_path = "/content/rag_pipeline/eval_results.json"
    with open(out_path, "w") as f:
        json.dump({
            "metrics": {
                "decision_accuracy"  : round(correct/n, 4),
                "hallucination_rate" : round(hallucin/n, 4),
                "over_answer_rate"   : round(over_ans/n, 4),
                "n_evaluated"        : n,
            },
            "results": results
        }, f, indent=2)
    print(f"\nSaved eval results → {out_path}")

    return results


# ── Run eval ─────────────────────────────────────────────────
eval_results = evaluate_rag(eval_samples, top_k=5, sleep_sec=0)


# ================================================================
# STEP 7 — SAMPLE INSPECTION (success + failure cases)
# ================================================================

def show_result(r):
    print(f"  ID        : {r['sample_id']}")
    print(f"  Source    : {r['source']}")
    print(f"  Query     : {r['query']}")
    print(f"  GT action : {r['gt_action']}")
    print(f"  Predicted : {r['pred_action']}  {'✅' if r['correct'] else '❌'}")
    print(f"  Response  : {r['response'][:150]}")
    print(f"  Retrieved : {r['top_retrieved']}")

print("\n" + "="*55)
print("CORRECT PREDICTIONS — 2 samples per action")
print("="*55)
for act in ["ANSWER","ASK","ABSTAIN"]:
    correct_cases = [r for r in eval_results
                     if r["correct"] and r["gt_action"]==act]
    print(f"\n  ── GT={act} ──")
    for r in correct_cases[:2]:
        show_result(r)
        print()

print("\n" + "="*55)
print("FAILURE CASES — hallucinations (predicted ANSWER, GT!=ANSWER)")
print("="*55)
failures = [r for r in eval_results
            if r["pred_action"]=="ANSWER" and r["gt_action"]!="ANSWER"]
for r in failures[:4]:
    show_result(r)
    print()

## **ENHANCED RAG**

In [ ]:
# ================================================================
# ENHANCED RAG — STEP A: SEMANTIC CHUNKING
# Sentence-boundary aware, no fixed word slicing
# ================================================================

!pip install nltk -q
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

def semantic_chunk(text, max_words=350, overlap_sents=2):
    """
    Chunk by sentence boundaries instead of hard word cuts.
    Overlap = carry last N sentences into next chunk.
    """
    sentences = sent_tokenize(text)
    if not sentences:
        return [text]

    chunks        = []
    current_sents = []
    current_wc    = 0

    for sent in sentences:
        sent_wc = len(sent.split())
        if current_wc + sent_wc > max_words and current_sents:
            chunks.append(" ".join(current_sents))
            # Overlap: carry last N sentences forward
            current_sents = current_sents[-overlap_sents:]
            current_wc    = sum(len(s.split()) for s in current_sents)
        current_sents.append(sent)
        current_wc += sent_wc

    if current_sents:
        chunks.append(" ".join(current_sents))

    return chunks if chunks else [text]


def needs_chunking(text, source):
    wc = len(text.split())
    if source == "contract_nli" and wc > 300: return True
    if source == "quac"         and wc > 400: return True
    return False


# Test
sample_text = " ".join([s["query"] for s in all_samples[:5]])
chunks = semantic_chunk(sample_text, max_words=30)
print(f"Semantic chunks produced: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  Chunk {i+1} ({len(c.split())} words): {c[:80]}...")

In [ ]:
# ================================================================
# ENHANCED RAG — STEP B: MULTI-GRANULARITY KB
# Each context stored at TWO levels:
#   Level 1 — full semantic chunk  (coarse)
#   Level 2 — individual sentences (fine)
# ================================================================

import os

# Ensure the output directory exists
os.makedirs("/content/rag_pipeline", exist_ok=True)

MG_KB_PATH    = "/content/rag_pipeline/mg_knowledge_base.json"
MG_FAISS_PATH = "/content/rag_pipeline/mg_faiss.index"
MG_EMB_PATH   = "/content/rag_pipeline/mg_embeddings.npy"

if os.path.exists(MG_KB_PATH):
    print("✅ Multi-granularity KB exists — loading...")
    with open(MG_KB_PATH) as f:
        mg_kb = json.load(f)
    print(f"   Total entries: {len(mg_kb)}")

else:
    print("Building multi-granularity KB...\n")

    ctx_groups = defaultdict(list)
    for s in all_samples:
        ctx_text = get_ctx_text(s)
        if ctx_text.strip():
            ctx_groups[hash_text(ctx_text)].append(s)

    mg_kb    = []
    seen     = {}
    entry_id = 0

    for ctx_hash, samples in ctx_groups.items():
        ctx_text  = get_ctx_text(samples[0])
        source    = samples[0]["metadata"]["source"]
        linked_ids     = [s["id"]     for s in samples]
        linked_actions = [s["action"] for s in samples]
        linked_queries = [s["query"]  for s in samples]
        action_dist    = dict(Counter(linked_actions))

        # ── Level 1: coarse semantic chunks ──────────────────
        coarse_chunks = (semantic_chunk(ctx_text)
                         if needs_chunking(ctx_text, source)
                         else [ctx_text])

        for c_idx, chunk in enumerate(coarse_chunks):
            ch = hash_text(chunk)
            if ch not in seen:
                seen[ch] = entry_id
                mg_kb.append({
                    "mg_id"          : f"mg_{entry_id:07d}",
                    "granularity"    : "coarse",
                    "parent_ctx_hash": ctx_hash,
                    "chunk_idx"      : c_idx,
                    "source"         : source,
                    "text"           : chunk,
                    "word_count"     : len(chunk.split()),
                    "linked_sample_ids"  : linked_ids,
                    "linked_actions"     : linked_actions,
                    "linked_queries"     : linked_queries,
                    "action_distribution": action_dist,
                })
                entry_id += 1

        # ── Level 2: fine sentence-level entries ──────────────
        sentences = sent_tokenize(ctx_text)
        for s_idx, sent in enumerate(sentences):
            sent = sent.strip()
            if len(sent.split()) < 5:   # skip trivially short sentences
                continue
            sh = hash_text(sent)
            if sh not in seen:
                seen[sh] = entry_id
                mg_kb.append({
                    "mg_id"          : f"mg_{entry_id:07d}",
                    "granularity"    : "fine",
                    "parent_ctx_hash": ctx_hash,
                    "sent_idx"       : s_idx,
                    "source"         : source,
                    "text"           : sent,
                    "word_count"     : len(sent.split()),
                    "linked_sample_ids"  : linked_ids,
                    "linked_actions"     : linked_actions,
                    "linked_queries"     : linked_queries,
                    "action_distribution": action_dist,
                })
                entry_id += 1

    print(f"✅ Multi-granularity KB: {len(mg_kb)} entries")
    coarse = sum(1 for e in mg_kb if e["granularity"]=="coarse")
    fine   = sum(1 for e in mg_kb if e["granularity"]=="fine")
    print(f"   Coarse chunks : {coarse}")
    print(f"   Fine sentences: {fine}")

    with open(MG_KB_PATH, "w") as f:
        json.dump(mg_kb, f, indent=2)
    print(f"Saved → {MG_KB_PATH}")


# ── Embed multi-granularity KB ────────────────────────────────
if os.path.exists(MG_FAISS_PATH):
    print("✅ MG FAISS exists — loading...")
    mg_embeddings = np.load(MG_EMB_PATH)
    mg_index      = faiss.read_index(MG_FAISS_PATH)
else:
    print(f"Embedding {len(mg_kb)} entries...")
    mg_texts      = [e["text"] for e in mg_kb]
    mg_embeddings = embedder.encode(
        mg_texts,
        batch_size           = 256,
        show_progress_bar    = True,
        convert_to_numpy     = True,
        normalize_embeddings = True
    )
    dim      = mg_embeddings.shape[1]
    mg_index = faiss.IndexFlatIP(dim)
    mg_index.add(mg_embeddings)

    np.save(MG_EMB_PATH, mg_embeddings)
    faiss.write_index(mg_index, MG_FAISS_PATH)
    print(f"✅ MG FAISS: {mg_index.ntotal} vectors")

In [ ]:
# ================================================================
# ENHANCED RAG — STEP C: HYBRID RETRIEVAL
# BM25 (sparse) + Dense (semantic) → score fusion
# ================================================================

!pip install rank_bm25 -q
from rank_bm25 import BM25Okapi

print("Building BM25 index...")
bm25_corpus  = [e["text"].lower().split() for e in mg_kb]
bm25_index   = BM25Okapi(bm25_corpus)
print(f"✅ BM25 index built over {len(bm25_corpus)} entries")


def hybrid_retrieve(query, top_k=5, alpha=0.5,
                    granularity_filter=None):
    """
    alpha=0.5 → equal weight BM25 + dense
    alpha=0.0 → pure BM25
    alpha=1.0 → pure dense
    granularity_filter: 'coarse' | 'fine' | None (both)
    """
    n = len(mg_kb)

    # ── Dense scores ─────────────────────────────────────────
    q_emb = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    dense_scores, dense_idxs = mg_index.search(q_emb, min(n, 200))
    dense_score_map = {
        int(idx): float(score)
        for idx, score in zip(dense_idxs[0], dense_scores[0])
        if idx >= 0
    }

    # ── BM25 scores ───────────────────────────────────────────
    bm25_raw    = bm25_index.get_scores(query.lower().split())
    # Normalize BM25 to [0,1]
    bm25_max    = bm25_raw.max() if bm25_raw.max() > 0 else 1.0
    bm25_norm   = bm25_raw / bm25_max

    # ── Fuse ──────────────────────────────────────────────────
    # Candidate pool = top-200 from dense ∪ top-200 from BM25
    bm25_top200 = np.argsort(bm25_raw)[::-1][:200]
    candidates  = set(dense_score_map.keys()) | set(bm25_top200.tolist())

    scored = []
    for idx in candidates:
        if idx >= n:
            continue
        entry = mg_kb[idx]
        if granularity_filter and entry["granularity"] != granularity_filter:
            continue
        d_score = dense_score_map.get(idx, 0.0)
        b_score = float(bm25_norm[idx])
        fused   = alpha * d_score + (1 - alpha) * b_score
        scored.append((fused, idx))

    scored.sort(key=lambda x: x[0], reverse=True)

    results = []
    for fused_score, idx in scored[:top_k]:
        e = mg_kb[idx]
        results.append({
            "mg_id"          : e["mg_id"],
            "granularity"    : e["granularity"],
            "source"         : e["source"],
            "fused_score"    : round(fused_score, 4),
            "dense_score"    : round(dense_score_map.get(idx,0.0), 4),
            "bm25_score"     : round(float(bm25_norm[idx]), 4),
            "text"           : e["text"],
            "action_dist"    : e["action_distribution"],
            "linked_queries" : e["linked_queries"][:2],
            "word_count"     : e["word_count"],
        })
    return results


# Quick test
print("\nHybrid retrieval test:")
hits = hybrid_retrieve("Can I file for bankruptcy?", top_k=3)
for h in hits:
    print(f"  [{h['granularity']:6}] fused={h['fused_score']} "
          f"dense={h['dense_score']} bm25={h['bm25_score']} "
          f"src={h['source']} | {h['text'][:80]}...")

In [ ]:
# ================================================================
# ENHANCED RAG — STEP D: QUERY UNDERSTANDING
# Rewrite vague queries + decompose multi-hop ones
# ================================================================

def rewrite_query(query):
    """
    Use Mistral to rewrite the query for better retrieval.
    Returns rewritten query string.
    """
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content":
          f"""Rewrite the following query to be more specific and retrieval-friendly.
Keep it as ONE concise sentence. Do not answer it.
Return ONLY the rewritten query, nothing else.

Original query: {query}
Rewritten query:"""}],
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=60,
                             do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens,
                            skip_special_tokens=True).strip()


def decompose_query(query):
    """
    Decompose a complex query into sub-queries.
    Only call this for queries that seem multi-hop.
    Returns list of sub-query strings.
    """
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content":
          f"""Decompose the following complex question into 2-3 simpler sub-questions
that can be answered independently.
Return ONLY the sub-questions, one per line, numbered.
If the question is already simple, return just: 1. {query}

Question: {query}
Sub-questions:"""}],
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100,
                             do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Parse numbered lines
    sub_qs = []
    for line in raw.split("\n"):
        line = line.strip()
        if line and line[0].isdigit() and "." in line:
            sub_q = line.split(".", 1)[-1].strip()
            if sub_q:
                sub_qs.append(sub_q)
    return sub_qs if sub_qs else [query]


def is_multihop(query):
    """Heuristic: multi-hop queries tend to be longer and
    contain comparison/relational keywords."""
    keywords = ["and", "both", "compare", "difference",
                "before", "after", "when did", "which of"]
    q_lower  = query.lower()
    return (len(query.split()) > 12 or
            any(k in q_lower for k in keywords))


# Test
q1 = "What are the environmental and economic differences between EVs and ICE vehicles?"
q2 = "Can I file for bankruptcy?"
print(f"Multi-hop? '{q1[:50]}...' → {is_multihop(q1)}")
print(f"Multi-hop? '{q2}' → {is_multihop(q2)}")
print(f"\nRewritten: {rewrite_query(q2)}")
print(f"\nDecomposed: {decompose_query(q1)}")

In [ ]:
# ================================================================
# ENHANCED RAG — STEP E: RE-RANKING + CONTEXT COMPRESSION
# ================================================================

!pip install sentence-transformers -q   # already installed but ensure
from sentence_transformers import CrossEncoder

print("Loading cross-encoder re-ranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Re-ranker loaded.")


def rerank(query, chunks, top_n=5):
    """Re-rank retrieved chunks using cross-encoder."""
    if not chunks:
        return chunks
    pairs  = [(query, c["text"][:512]) for c in chunks]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, chunks),
                    key=lambda x: x[0], reverse=True)
    reranked = []
    for score, chunk in ranked[:top_n]:
        chunk["rerank_score"] = round(float(score), 4)
        reranked.append(chunk)
    return reranked


def compress_context(query, chunks, max_sents_per_chunk=3):
    """
    Keep only the most relevant sentences from each chunk.
    Uses BM25 sentence-level scoring against the query.
    """
    query_words   = query.lower().split()
    compressed    = []

    for chunk in chunks:
        sentences = sent_tokenize(chunk["text"])
        if len(sentences) <= max_sents_per_chunk:
            compressed.append(chunk)
            continue

        # Score each sentence via keyword overlap
        sent_scores = []
        for sent in sentences:
            sw    = sent.lower().split()
            score = sum(1 for w in query_words if w in sw)
            sent_scores.append((score, sent))

        # Keep top N sentences, preserve original order
        top_sents = sorted(sent_scores,
                           key=lambda x: x[0], reverse=True
                           )[:max_sents_per_chunk]
        top_texts = {s[1] for s in top_sents}
        ordered   = [s for s in sentences if s in top_texts]

        compressed_chunk          = dict(chunk)
        compressed_chunk["text"]  = " ".join(ordered)
        compressed_chunk["compressed"] = True
        compressed.append(compressed_chunk)

    return compressed


# Test
test_chunks = hybrid_retrieve("bankruptcy filing rules", top_k=10)
reranked    = rerank("bankruptcy filing rules", test_chunks, top_n=5)
compressed  = compress_context("bankruptcy filing rules", reranked)

print("Re-ranking + compression test:")
for i, c in enumerate(compressed, 1):
    print(f"  {i}. rerank={c.get('rerank_score','—')} "
          f"| compressed={c.get('compressed', False)} "
          f"| words={len(c['text'].split())} "
          f"| {c['text'][:80]}...")

In [ ]:
# ================================================================
# ENHANCED RAG — STEP F: FULL PIPELINE + SELF-REFLECTION
# ================================================================

SYSTEM_PROMPT_ENHANCED = """You are a decision-aware intelligent assistant.

Given a user query and retrieved context passages, you must:

1. Carefully read the context.
2. Decide which action is appropriate:
   - ANSWER   → sufficient information present
   - ASK      → key information missing, clarification needed
   - ABSTAIN  → cannot be answered from context at all

3. Output in this EXACT format:
ACTION: <ANSWER|ASK|ABSTAIN>
RESPONSE: <answer, clarification question, or abstain statement>

Rules:
- Never hallucinate. If uncertain → ASK or ABSTAIN.
- If ASKing, ask ONE focused clarification question.
- Be concise."""

REFLECTION_PROMPT = """You generated this response to a query.
Verify if your action is correct given the context.

Query: {query}
Your action: {action}
Your response: {response}
Context used: {context_preview}

Is your action correct?
- If ANSWER: is it supported by context? If not → change to ABSTAIN
- If ASK: is clarification truly needed? If context is sufficient → change to ANSWER
- If ABSTAIN: is context truly insufficient?

Output ONLY:
VERIFIED_ACTION: <ANSWER|ASK|ABSTAIN>
VERIFIED_RESPONSE: <corrected response if needed, else same>"""


def build_enhanced_prompt(query, chunks, history=None):
    ctx_block = ""
    for i, c in enumerate(chunks, 1):
        ctx_block += (f"\n[Context {i} | src={c['source']} "
                      f"| gran={c.get('granularity','—')} "
                      f"| score={c.get('rerank_score', c.get('fused_score','—'))}]\n")
        ctx_block += c["text"][:600] + "\n"

    history_block = ""
    if history:
        history_block = "Conversation so far:\n"
        for t in history:
            history_block += f"  User: {t['query']}\n"
            history_block += (f"  Assistant [{t['action']}]: "
                              f"{t['response']}\n")
        history_block += "\n"

    content = (f"{history_block}Query:\n{query}\n\n"
               f"Retrieved Context:\n{ctx_block}")
    messages = [{"role": "user",
                 "content": SYSTEM_PROMPT_ENHANCED + "\n\n" + content}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)


def llm_generate(prompt, max_new_tokens=200):
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=3072
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = 1.0,
            pad_token_id   = tokenizer.eos_token_id,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True).strip()


def parse_action_response(raw, action_key="ACTION:",
                           response_key="RESPONSE:"):
    action   = "ABSTAIN"
    resp_txt = raw
    for line in raw.split("\n"):
        ls = line.strip()
        if ls.startswith(action_key):
            a = ls.replace(action_key,"").strip()
            if a in ["ANSWER","ASK","ABSTAIN"]:
                action = a
        if ls.startswith(response_key):
            resp_txt = ls.replace(response_key,"").strip()
    return action, resp_txt


def enhanced_rag_infer(query, top_k_retrieve=20, top_k_rerank=5,
                       alpha=0.5, history=None,
                       use_rewrite=True,
                       use_decompose=True,
                       use_reflection=True):
    """
    Full enhanced RAG:
    rewrite → decompose → hybrid retrieve → rerank
    → compress → generate → reflect
    """
    trace = {"original_query": query}

    # 1. Query rewriting
    working_query = query
    if use_rewrite:
        working_query        = rewrite_query(query)
        trace["rewritten_q"] = working_query

    # 2. Conditional decomposition
    all_chunks = []
    if use_decompose and is_multihop(query):
        sub_qs              = decompose_query(query)
        trace["sub_queries"] = sub_qs
        for sq in sub_qs:
            all_chunks.extend(
                hybrid_retrieve(sq, top_k=top_k_retrieve//len(sub_qs),
                                alpha=alpha)
            )
        # Deduplicate by mg_id
        seen_ids   = set()
        dedup      = []
        for c in all_chunks:
            if c["mg_id"] not in seen_ids:
                seen_ids.add(c["mg_id"])
                dedup.append(c)
        all_chunks = dedup
    else:
        all_chunks = hybrid_retrieve(working_query,
                                     top_k=top_k_retrieve,
                                     alpha=alpha)

    trace["num_retrieved"] = len(all_chunks)

    # 3. Re-rank
    reranked = rerank(working_query, all_chunks, top_n=top_k_rerank)
    trace["num_reranked"] = len(reranked)

    # 4. Compress
    compressed = compress_context(working_query, reranked)

    # 5. Generate
    prompt     = build_enhanced_prompt(query, compressed, history)
    raw_output = llm_generate(prompt)
    action, response = parse_action_response(raw_output)
    trace["first_action"]   = action
    trace["first_response"] = response

    # 6. Self-reflection
    if use_reflection:
        ctx_preview = " | ".join(c["text"][:100] for c in compressed[:2])
        ref_prompt  = tokenizer.apply_chat_template(
            [{"role": "user",
              "content": REFLECTION_PROMPT.format(
                  query           = query,
                  action          = action,
                  response        = response,
                  context_preview = ctx_preview
              )}],
            tokenize=False, add_generation_prompt=True
        )
        ref_raw   = llm_generate(ref_prompt, max_new_tokens=150)
        v_action, v_response = parse_action_response(
            ref_raw,
            action_key   = "VERIFIED_ACTION:",
            response_key = "VERIFIED_RESPONSE:"
        )
        if v_action != action:
            trace["reflection_changed"] = True
            action, response = v_action, v_response
        else:
            trace["reflection_changed"] = False

    trace["final_action"]   = action
    trace["final_response"] = response

    return {
        "query"            : query,
        "action"           : action,
        "response"         : response,
        "raw_output"       : raw_output,
        "retrieved_chunks" : compressed,
        "num_retrieved"    : len(compressed),
        "trace"            : trace,
    }


# ── Quick test ────────────────────────────────────────────────
print("Enhanced RAG test:")
test_q = "Can I file for bankruptcy under Chapter 9?"
out    = enhanced_rag_infer(test_q, use_rewrite=True,
                             use_decompose=True, use_reflection=True)
print(f"Query     : {out['query']}")
print(f"Action    : {out['action']}")
print(f"Response  : {out['response']}")
print(f"Trace     : {json.dumps(out['trace'], indent=2)}")

In [ ]:
!pip install tqdm -q
from tqdm import tqdm

In [ ]:
# ================================================================
# ENHANCED RAG — STEP G: COMPARATIVE EVALUATION
# Baseline RAG vs Enhanced RAG on same eval set
# ================================================================

def evaluate_enhanced_rag(eval_samples, top_k_retrieve=20,
                           top_k_rerank=5):
    results  = []
    correct  = 0
    hallucin = 0
    reflections_changed = 0

    for i, sample in enumerate(tqdm(eval_samples, desc="Evaluating")):
        if i % 20 == 0:
            print(f"  {i+1}/{len(eval_samples)}...")
        try:
            out = enhanced_rag_infer(
                sample["query"],
                top_k_retrieve = top_k_retrieve,
                top_k_rerank   = top_k_rerank,
                use_rewrite    = True,
                use_decompose  = True,
                use_reflection = True,
            )
        except Exception as ex:
            print(f"  Error on {i}: {ex}")
            out = {"action":"ABSTAIN","response":"error",
                   "retrieved_chunks":[],"trace":{}}

        gt          = sample["action"]
        pred        = out["action"]
        is_correct  = (pred == gt)
        if is_correct: correct  += 1
        if pred == "ANSWER" and gt != "ANSWER": hallucin += 1
        if out["trace"].get("reflection_changed", False):
            reflections_changed += 1

        results.append({
            "sample_id"          : sample["id"],
            "source"             : sample["metadata"]["source"],
            "query"              : sample["query"],
            "gt_action"          : gt,
            "gt_response"        : sample["response"],          # ← ground truth
            "gt_failure_mode"    : sample["state"]["failure_mode"],
            "gt_difficulty"      : sample["state"]["difficulty"],
            "pred_action"        : pred,
            "pred_response"      : out["response"],
            "correct"            : is_correct,
            "rewritten_query"    : out["trace"].get("rewritten_q",""),
            "reflection_changed" : out["trace"].get("reflection_changed",False),
            "top_chunks"         : [
                {
                    "source"      : c["source"],
                    "granularity" : c.get("granularity","—"),
                    "score"       : c.get("rerank_score",
                                    c.get("fused_score","—")),
                    "text_preview": c["text"][:120]
                }
                for c in out.get("retrieved_chunks", [])
            ],
        })

    n = len(results)
    print("\n" + "="*55)
    print("  ENHANCED RAG RESULTS")
    print("="*55)
    print(f"  Total evaluated       : {n}")
    print(f"  Decision Accuracy     : {correct/n*100:.1f}%")
    print(f"  Hallucination Rate    : {hallucin/n*100:.1f}%")
    print(f"  Reflection changes    : {reflections_changed} "
          f"({100*reflections_changed/n:.1f}% of samples)")

    print(f"\n  --- Per-action accuracy ---")
    for act in ["ANSWER","ASK","ABSTAIN"]:
        act_res  = [r for r in results if r["gt_action"]==act]
        if not act_res: continue
        act_corr = sum(r["correct"] for r in act_res)
        print(f"  {act:10}: {act_corr}/{len(act_res)} "
              f"({100*act_corr/len(act_res):.1f}%)")

    print(f"\n  --- Confusion (GT → Predicted) ---")
    confusion = defaultdict(Counter)
    for r in results:
        confusion[r["gt_action"]][r["pred_action"]] += 1
    for gt, preds in sorted(confusion.items()):
        print(f"  GT={gt:10} → {dict(preds)}")

    # Save
    enhanced_eval_path = "/content/rag_pipeline/enhanced_eval_results.json"
    with open(enhanced_eval_path, "w") as f:
        json.dump({
            "metrics": {
                "decision_accuracy"    : round(correct/n, 4),
                "hallucination_rate"   : round(hallucin/n, 4),
                "reflection_changes"   : reflections_changed,
                "n_evaluated"          : n,
            },
            "results": results
        }, f, indent=2)
    print(f"\nSaved → {enhanced_eval_path}")
    return results

    print(f"\n  --- Sample predictions (first 5) ---")
    for r in results[:5]:
        match = "✅" if r["correct"] else "❌"
        print(f"\n  {match} ID: {r['sample_id']}")
        print(f"     Query      : {r['query'][:80]}")
        print(f"     GT action  : {r['gt_action']:10} | GT response : {r['gt_response'][:80]}")
        print(f"     PRED action: {r['pred_action']:10} | PRED response: {r['pred_response'][:80]}")



# ── Compare baseline vs enhanced ─────────────────────────────
def compare_results(baseline_path, enhanced_path):
    with open(baseline_path)  as f: base = json.load(f)
    with open(enhanced_path)  as f: enh  = json.load(f)

    print("\n" + "="*55)
    print("  BASELINE vs ENHANCED RAG COMPARISON")
    print("="*55)
    print(f"  {'Metric':30} {'Baseline':>10} {'Enhanced':>10}")
    print(f"  {'-'*50}")

    for key in ["decision_accuracy","hallucination_rate"]:
        bv = base["metrics"].get(key, 0)
        ev = enh["metrics"].get(key, 0)
        delta = ev - bv
        sign  = "+" if delta > 0 else ""
        print(f"  {key:30} {bv*100:>9.1f}% {ev*100:>9.1f}%  "
              f"({sign}{delta*100:.1f}%)")


eval_samples = sample_eval_set(all_samples, n_per_action=100)
print(f"Eval set: {len(eval_samples)} | "
      f"{Counter(s['action'] for s in eval_samples)}")

enhanced_results = evaluate_enhanced_rag(eval_samples)

In [ ]:
# ================================================================
# INTERACTIVE RAG DEBUG WINDOW
# ================================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Widgets ───────────────────────────────────────────────────
query_input = widgets.Textarea(
    placeholder = "Type your query here...",
    layout      = widgets.Layout(width="700px", height="80px")
)

top_k_slider = widgets.IntSlider(
    value=10, min=5, max=50, step=5,
    description="Retrieve K:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="400px")
)

rerank_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="Rerank to:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="400px")
)

alpha_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.1,
    description="BM25↔Dense α:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px"),
    readout_format=".1f"
)

rewrite_toggle    = widgets.Checkbox(value=True,  description="Query Rewrite")
decompose_toggle  = widgets.Checkbox(value=True,  description="Decompose")
reflection_toggle = widgets.Checkbox(value=True,  description="Self-Reflect")

run_button = widgets.Button(
    description   = "▶ Run RAG",
    button_style  = "primary",
    layout        = widgets.Layout(width="150px", height="40px")
)

out = widgets.Output()

ACTION_COLOR = {
    "ANSWER" : "#00e676",
    "ASK"    : "#40c4ff",
    "ABSTAIN": "#ff5252"
}

def render_debug(result):
    trace  = result["trace"]
    action = result["action"]
    color  = ACTION_COLOR.get(action, "#ffffff")
    chunks = result.get("retrieved_chunks", [])

    # ── Trace section ─────────────────────────────────────────
    rewrite_row = ""
    if "rewritten_q" in trace:
        rewrite_row = f"""
        <tr>
          <td style="color:#90caf9; padding:4px;"><b>Rewritten Query</b></td>
          <td style="color:#fff176; padding:4px;">{trace['rewritten_q']}</td>
        </tr>"""

    decompose_row = ""
    if "sub_queries" in trace:
        sqs = "<br>".join(f"• {q}" for q in trace["sub_queries"])
        decompose_row = f"""
        <tr>
          <td style="color:#90caf9; padding:4px;"><b>Sub-queries</b></td>
          <td style="color:#ffcc80; padding:4px;">{sqs}</td>
        </tr>"""

    reflection_row = f"""
        <tr>
          <td style="color:#90caf9; padding:4px;"><b>Reflection changed?</b></td>
          <td style="color:{'#ff5252' if trace.get('reflection_changed') else '#00e676'}; padding:4px;">
            {'YES — action was corrected' if trace.get('reflection_changed') else 'No change'}
          </td>
        </tr>"""

    # ── Chunks section ────────────────────────────────────────
    chunk_rows = ""
    for i, c in enumerate(chunks, 1):
        score     = c.get("rerank_score", c.get("fused_score","—"))
        gran_col  = "#80cbc4" if c.get("granularity")=="coarse" else "#ce93d8"
        chunk_rows += f"""
        <tr style="border-bottom:1px solid #333;">
          <td style="padding:6px; color:#aaa; vertical-align:top;">{i}</td>
          <td style="padding:6px; vertical-align:top;">
            <span style="color:{gran_col}; font-size:11px;">
              [{c.get('granularity','—')}]
            </span>
            <span style="color:#80deea;"> {c['source']}</span>
          </td>
          <td style="padding:6px; color:#ffcc80; vertical-align:top;">{score}</td>
          <td style="padding:6px; color:#cfd8dc; font-size:11px;
                     vertical-align:top; white-space:pre-wrap;">{c['text'][:300]}...</td>
        </tr>"""

    # ── First vs final action diff ────────────────────────────
    first_action  = trace.get("first_action", action)
    first_resp    = trace.get("first_response", result["response"])
    first_color   = ACTION_COLOR.get(first_action, "#fff")
    changed_badge = (
        f'<span style="color:#ff5252; font-size:11px;"> ⟶ changed to '
        f'<b style="color:{color};">{action}</b> after reflection</span>'
        if trace.get("reflection_changed") else ""
    )

    html = f"""
    <div style="font-family:monospace; font-size:13px; background:#1e1e1e;
                color:#e0e0e0; padding:16px; border-radius:8px;
                border:1px solid #444;">

      <!-- Header -->
      <div style="margin-bottom:10px;">
        <b style="color:#90caf9; font-size:14px;">🔍 RAG Debug View</b>
        &nbsp;&nbsp;
        <span style="color:#aaa; font-size:11px;">
          retrieved={trace.get('num_retrieved','—')} →
          reranked={trace.get('num_reranked','—')}
        </span>
      </div>
      <hr style="border:0.5px solid #444;">

      <!-- Query trace -->
      <table style="width:100%; border-collapse:collapse; margin-bottom:10px;">
        <tr>
          <td style="color:#90caf9; padding:4px; width:160px;"><b>Original Query</b></td>
          <td style="color:#fff176; padding:4px;">{result['query']}</td>
        </tr>
        {rewrite_row}
        {decompose_row}
        {reflection_row}
      </table>
      <hr style="border:0.5px solid #444;">

      <!-- Retrieved chunks -->
      <div style="margin-bottom:8px;">
        <b style="color:#ce93d8;">📄 Retrieved & Compressed Chunks</b>
      </div>
      <table style="width:100%; border-collapse:collapse;
                    font-size:12px; margin-bottom:10px;">
        <tr style="color:#90caf9; border-bottom:1px solid #555;">
          <th style="padding:4px; text-align:left;">#</th>
          <th style="padding:4px; text-align:left;">Granularity | Source</th>
          <th style="padding:4px; text-align:left;">Score</th>
          <th style="padding:4px; text-align:left;">Text</th>
        </tr>
        {chunk_rows}
      </table>
      <hr style="border:0.5px solid #444;">

      <!-- Generation -->
      <div style="margin-bottom:6px;">
        <b style="color:#90caf9;">⚡ First Generation</b>
        &nbsp;
        <span style="color:{first_color}; font-weight:bold;">
          {first_action}
        </span>
        {changed_badge}
      </div>
      <div style="background:#263238; padding:8px; border-radius:4px;
                  color:#cfd8dc; font-size:12px; margin-bottom:10px;">
        {first_resp}
      </div>

      <!-- Final output -->
      <div style="margin-bottom:6px;">
        <b style="color:#90caf9;">🎯 Final Action:</b>
        <span style="color:{color}; font-weight:bold; font-size:15px;">
          ● {action}
        </span>
      </div>
      <div style="background:#1b3a2b; padding:10px; border-radius:4px;
                  border-left:3px solid {color}; color:#c8e6c9;
                  font-size:13px;">
        {result['response']}
      </div>

    </div>
    """
    return html


def on_run(b):
    query = query_input.value.strip()
    if not query:
        with out:
            clear_output()
            print("Please enter a query.")
        return

    with out:
        clear_output(wait=True)
        print("⏳ Running enhanced RAG pipeline...")

    try:
        result = enhanced_rag_infer(
            query,
            top_k_retrieve  = top_k_slider.value,
            top_k_rerank    = rerank_slider.value,
            alpha           = alpha_slider.value,
            use_rewrite     = rewrite_toggle.value,
            use_decompose   = decompose_toggle.value,
            use_reflection  = reflection_toggle.value,
        )
        with out:
            clear_output(wait=True)
            display(HTML(render_debug(result)))

    except Exception as ex:
        with out:
            clear_output(wait=True)
            print(f"❌ Error: {ex}")
            import traceback
            traceback.print_exc()


run_button.on_click(on_run)

# ── Layout ────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HTML("<b style='color:#90caf9; font-size:14px;'>"
                 "🧪 Interactive RAG Debugger</b>"),
    query_input,
    widgets.HBox([top_k_slider, rerank_slider]),
    widgets.HBox([alpha_slider]),
    widgets.HBox([rewrite_toggle, decompose_toggle, reflection_toggle]),
    run_button,
    out
]))

## **DECISION-AWARE RAG**

In [ ]:
# ================================================================
# DECISION-AWARE RAG v3 — SECTION 1: EVIDENCE SCORING
# Computes 3 explicit signals before any generation happens
# ================================================================

import numpy as np
from sentence_transformers import CrossEncoder, SentenceTransformer
from nltk.tokenize import sent_tokenize

# Reranker already loaded as `reranker` from previous step
# Embedder already loaded as `embedder`

def compute_evidence_scores(query, chunks):
    """
    Returns 3 signals:
      confidence_score : how strongly top chunk matches query
      coverage_score   : how much of query is covered across chunks
      score_gap        : difference between top and second chunk score
                         (low gap = ambiguous retrieval)
    """
    if not chunks:
        return {
            "confidence_score": 0.0,
            "coverage_score"  : 0.0,
            "score_gap"       : 0.0,
            "mean_score"      : 0.0,
            "num_chunks"      : 0
        }

    scores = [c.get("rerank_score",
               c.get("fused_score", 0.0)) for c in chunks]

    # Normalize reranker scores to [0,1]
    # Cross-encoder outputs raw logits — sigmoid to normalize
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    norm_scores = [sigmoid(s) if s > 1 or s < 0
                   else s for s in scores]

    confidence = float(np.max(norm_scores))
    mean_score = float(np.mean(norm_scores))
    score_gap  = float(norm_scores[0] - norm_scores[1]) \
                 if len(norm_scores) > 1 else confidence

    # Coverage: what fraction of query words appear in top chunks
    query_words   = set(query.lower().split())
    stop_words    = {"is","the","a","an","of","in","and","or",
                     "to","it","that","this","for","on","at"}
    content_words = query_words - stop_words
    if not content_words:
        coverage = confidence
    else:
        all_chunk_text = " ".join(c["text"].lower() for c in chunks)
        covered = sum(1 for w in content_words if w in all_chunk_text)
        coverage = covered / len(content_words)

    return {
        "confidence_score": round(confidence, 4),
        "coverage_score"  : round(coverage, 4),
        "score_gap"       : round(score_gap, 4),
        "mean_score"      : round(mean_score, 4),
        "num_chunks"      : len(chunks)
    }


def compute_ambiguity_score(query):
    """
    Heuristic ambiguity signals — no LLM needed.
    Returns score ∈ [0,1] where 1 = very ambiguous.
    """
    signals   = []
    q_lower   = query.lower().strip()
    q_words   = q_lower.split()

    # 1. Very short queries are likely underspecified
    signals.append(1.0 if len(q_words) <= 4 else 0.0)

    # 2. Dangling pronouns with no clear referent
    pronouns  = {"it","its","they","their","this","that","these",
                 "those","he","she","him","her"}
    has_pronoun = any(w in pronouns for w in q_words)
    signals.append(0.8 if has_pronoun else 0.0)

    # 3. Vague quantifiers / generic terms
    vague     = {"something","anything","everything","some",
                 "any","many","various","certain","related"}
    has_vague = any(w in vague for w in q_words)
    signals.append(0.6 if has_vague else 0.0)

    # 4. Missing entity — no proper noun or named entity signal
    has_caps  = any(w[0].isupper() for w in query.split()
                    if len(w) > 2 and w not in
                    {"Can","Does","Is","Are","What","Who",
                     "When","Where","Why","How"})
    signals.append(0.4 if not has_caps else 0.0)

    # 5. Comparative without both sides specified
    comp_kw   = {"better","worse","difference","compare",
                 "vs","versus","between"}
    has_comp  = any(w in comp_kw for w in q_words)
    q_entities = [w for w in q_words if w[0].isupper()] \
                 if not q_lower.startswith(q_words[0]) else []
    if has_comp and len(q_entities) < 2:
        signals.append(0.7)
    else:
        signals.append(0.0)

    return round(float(np.mean(signals)), 4)


def check_context_conflict(chunks):
    """
    Detect if retrieved chunks contradict each other.
    Simple: compute pairwise cosine similarity of chunk embeddings.
    Very low similarity between top chunks = potential conflict.
    Returns conflict_score ∈ [0,1] where 1 = high conflict.
    """
    if len(chunks) < 2:
        return 0.0

    texts = [c["text"][:300] for c in chunks[:4]]
    embs  = embedder.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # Pairwise cosine (already normalized → dot product)
    sims = []
    for i in range(len(embs)):
        for j in range(i+1, len(embs)):
            sims.append(float(np.dot(embs[i], embs[j])))

    avg_sim       = np.mean(sims)
    # Low average similarity between chunks = conflict signal
    conflict_score = round(max(0.0, 1.0 - avg_sim), 4)
    return conflict_score


# Quick test
test_chunks = hybrid_retrieve("Can I file for bankruptcy?", top_k=10)
reranked    = rerank("Can I file for bankruptcy?", test_chunks, top_n=5)
ev_scores   = compute_evidence_scores("Can I file for bankruptcy?",
                                       reranked)
amb_score   = compute_ambiguity_score("Can I file for bankruptcy?")
conf_score  = check_context_conflict(reranked)

print("Evidence scores:")
for k, v in ev_scores.items():
    print(f"  {k:20}: {v}")
print(f"  {'ambiguity_score':20}: {amb_score}")
print(f"  {'conflict_score':20}: {conf_score}")

In [ ]:
# ================================================================
# SECTION 2: ANSWERABILITY CLASSIFIER
# Separate focused LLM call — decide BEFORE generating
# ================================================================

ANSWERABILITY_PROMPT = """You are an evidence evaluator.

Given a query and retrieved context, evaluate ONLY whether the context
contains sufficient information.

Output EXACTLY one of:
ANSWERABLE        - context clearly supports answering the query
NEEDS_CLARIFICATION - query is ambiguous or missing key information
NOT_ANSWERABLE    - context lacks the information needed

Then output one line:
REASON: <one short sentence why>

Do NOT answer the query. Only evaluate answerability.

Query: {query}

Context:
{context}

Output:"""


def answerability_classify(query, chunks):
    """
    Dedicated LLM call just for decision — no generation.
    Returns: label, reason, raw_output
    """
    # Use only top 3 chunks, truncated — keep this call cheap
    ctx = ""
    for i, c in enumerate(chunks[:3], 1):
        ctx += f"[{i}] {c['text'][:400]}\n"

    prompt = tokenizer.apply_chat_template(
        [{"role": "user",
          "content": ANSWERABILITY_PROMPT.format(
              query=query, context=ctx)}],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=1536
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 80,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    new_toks   = out[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(new_toks,
                                  skip_special_tokens=True).strip()

    # Parse
    label  = "NOT_ANSWERABLE"   # safe default
    reason = ""

    for line in raw_output.split("\n"):
        ls = line.strip()
        if "ANSWERABLE" in ls and "NOT" not in ls \
                and "NEEDS" not in ls:
            label = "ANSWERABLE"
        elif "NEEDS_CLARIFICATION" in ls:
            label = "NEEDS_CLARIFICATION"
        elif "NOT_ANSWERABLE" in ls:
            label = "NOT_ANSWERABLE"
        if ls.startswith("REASON:"):
            reason = ls.replace("REASON:","").strip()

    return {
        "label"     : label,
        "reason"    : reason,
        "raw_output": raw_output
    }


# Test
result = answerability_classify(
    "Can I file for bankruptcy under Chapter 9?",
    reranked
)
print(f"Label  : {result['label']}")
print(f"Reason : {result['reason']}")

In [ ]:
# ================================================================
# SECTION 3: HARD GATING DECISION MODULE
# Combines all signals → hard threshold decision
# No LLM casual deciding — rule-based gating
# ================================================================

# ── Thresholds (tunable) ─────────────────────────────────────
TAU_CONFIDENCE  = 0.35   # below → ABSTAIN
TAU_COVERAGE    = 0.30   # below → ABSTAIN
TAU_AMBIGUITY   = 0.45   # above → ASK
TAU_CONFLICT    = 0.70   # above → ASK or ABSTAIN

def hard_gate_decision(ev_scores, ambiguity_score,
                       conflict_score, classifier_label):
    """
    Explicit rule-based gating.
    Priority order:
      1. Conflict → ASK (context is contradictory)
      2. Low confidence + low coverage → ABSTAIN
      3. High ambiguity → ASK
      4. Classifier says NOT_ANSWERABLE → ABSTAIN
      5. Classifier says NEEDS_CLARIFICATION → ASK
      6. Otherwise → ANSWER

    Returns: action, gate_reason, signals_used
    """
    confidence = ev_scores["confidence_score"]
    coverage   = ev_scores["coverage_score"]
    signals    = {
        "confidence"        : confidence,
        "coverage"          : coverage,
        "ambiguity"         : ambiguity_score,
        "conflict"          : conflict_score,
        "classifier_label"  : classifier_label
    }

    # Rule 1 — High conflict in retrieved chunks
    if conflict_score > TAU_CONFLICT:
        return "ASK", \
               f"Context conflict detected (score={conflict_score})", \
               signals

    # Rule 2 — Low retrieval confidence AND low coverage
    if confidence < TAU_CONFIDENCE and coverage < TAU_COVERAGE:
        return "ABSTAIN", \
               f"Insufficient evidence: confidence={confidence}, " \
               f"coverage={coverage}", \
               signals

    # Rule 3 — High ambiguity in query
    if ambiguity_score > TAU_AMBIGUITY:
        return "ASK", \
               f"Query ambiguous (score={ambiguity_score})", \
               signals

    # Rule 4 — Classifier explicitly says not answerable
    if classifier_label == "NOT_ANSWERABLE":
        return "ABSTAIN", \
               "Answerability classifier: NOT_ANSWERABLE", \
               signals

    # Rule 5 — Classifier says needs clarification
    if classifier_label == "NEEDS_CLARIFICATION":
        return "ASK", \
               "Answerability classifier: NEEDS_CLARIFICATION", \
               signals

    # Rule 6 — Default: answer
    return "ANSWER", \
           f"Sufficient evidence: confidence={confidence}, " \
           f"coverage={coverage}", \
           signals


# Test
action, reason, signals = hard_gate_decision(
    ev_scores, amb_score, conf_score,
    result["label"]
)
print(f"Decision : {action}")
print(f"Reason   : {reason}")
print(f"Signals  : {signals}")

In [ ]:
# ================================================================
# SECTION 4: ACTION-SPECIFIC GENERATORS
# Separate prompt per action — no single generic prompt
# ================================================================

ANSWER_PROMPT = """You are a precise factual assistant.

The query has been verified as answerable from the context below.
Answer DIRECTLY and CONCISELY using only the provided context.
Do not add information not present in the context.

Query: {query}

Context:
{context}

Answer:"""

ASK_PROMPT = """You are a clarification assistant.

The query cannot be fully answered because: {reason}

Generate exactly ONE focused clarification question that, if answered,
would allow a complete response to the original query.

Do NOT attempt to answer the query.
Output ONLY the clarification question, nothing else.

Original query: {query}

Clarification question:"""

ABSTAIN_PROMPT = """You are an honest assistant.

The query cannot be answered because: {reason}

Generate a brief, honest statement explaining why you cannot answer.
Be specific about what information is missing.
Do not make up information.

Query: {query}

Statement:"""


def generate_answer(query, chunks, reason=""):
    ctx = ""
    for i, c in enumerate(chunks[:5], 1):
        ctx += f"[{i}] {c['text'][:500]}\n"

    prompt = tokenizer.apply_chat_template(
        [{"role": "user",
          "content": ANSWER_PROMPT.format(
              query=query, context=ctx)}],
        tokenize=False, add_generation_prompt=True
    )
    return _generate(prompt, max_new_tokens=200)


def generate_clarification(query, reason=""):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user",
          "content": ASK_PROMPT.format(
              query=query, reason=reason)}],
        tokenize=False, add_generation_prompt=True
    )
    raw = _generate(prompt, max_new_tokens=80)
    # Ensure output ends with "?"
    raw = raw.strip()
    if not raw.endswith("?"):
        # Find last sentence and add ?
        sentences = raw.split(".")
        for s in reversed(sentences):
            s = s.strip()
            if len(s) > 5:
                raw = s + "?"
                break
    return raw


def generate_abstain(query, reason=""):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user",
          "content": ABSTAIN_PROMPT.format(
              query=query, reason=reason)}],
        tokenize=False, add_generation_prompt=True
    )
    return _generate(prompt, max_new_tokens=100)


def _generate(prompt, max_new_tokens=200):
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=2048
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks,
                            skip_special_tokens=True).strip()


# Quick test
print("Answer  :", generate_answer(
    "Can I file bankruptcy under Chapter 9?", reranked)[:120])
print("Ask     :", generate_clarification(
    "Tell me about its impact",
    reason="'its' has no clear referent"))
print("Abstain :", generate_abstain(
    "What is the GDP of XYZ country?",
    reason="No evidence found in context"))

In [ ]:
# ================================================================
# SECTION 5: FULL DECISION-AWARE RAG v3 PIPELINE
# retrieve → score → classify → hard-gate → action-specific generate
# ================================================================

def decision_aware_rag(query, top_k_retrieve=20, top_k_rerank=5,
                       alpha=0.5, history=None,
                       use_rewrite=True, use_decompose=True):
    """
    v3 Pipeline:
    1. Query rewrite + decompose
    2. Hybrid retrieval
    3. Re-rank + compress
    4. Evidence scoring (confidence, coverage, ambiguity, conflict)
    5. Answerability classifier (separate LLM call)
    6. Hard gating → action decision
    7. Action-specific generation
    """
    trace = {"original_query": query}

    # ── 1. Query understanding ────────────────────────────────
    working_query = query
    if use_rewrite:
        working_query        = rewrite_query(query)
        trace["rewritten_q"] = working_query

    all_chunks = []
    if use_decompose and is_multihop(query):
        sub_qs               = decompose_query(query)
        trace["sub_queries"] = sub_qs
        for sq in sub_qs:
            all_chunks.extend(
                hybrid_retrieve(sq,
                    top_k=top_k_retrieve//max(len(sub_qs),1),
                    alpha=alpha)
            )
        seen_ids = set()
        dedup    = []
        for c in all_chunks:
            if c["mg_id"] not in seen_ids:
                seen_ids.add(c["mg_id"])
                dedup.append(c)
        all_chunks = dedup
    else:
        all_chunks = hybrid_retrieve(working_query,
                                     top_k=top_k_retrieve,
                                     alpha=alpha)

    # ── 2. Re-rank + compress ─────────────────────────────────
    reranked   = rerank(working_query, all_chunks,
                        top_n=top_k_rerank)
    compressed = compress_context(working_query, reranked)
    trace["num_retrieved"] = len(all_chunks)
    trace["num_reranked"]  = len(reranked)

    # ── 3. Evidence scoring ───────────────────────────────────
    ev_scores      = compute_evidence_scores(query, compressed)
    ambiguity      = compute_ambiguity_score(query)
    conflict       = check_context_conflict(compressed)

    trace["evidence_scores"] = ev_scores
    trace["ambiguity_score"] = ambiguity
    trace["conflict_score"]  = conflict

    # ── 4. Answerability classifier ───────────────────────────
    classifier     = answerability_classify(query, compressed)
    trace["classifier_label"]  = classifier["label"]
    trace["classifier_reason"] = classifier["reason"]

    # ── 5. Hard gating ────────────────────────────────────────
    action, gate_reason, signals = hard_gate_decision(
        ev_scores, ambiguity, conflict, classifier["label"]
    )
    trace["gate_decision"] = action
    trace["gate_reason"]   = gate_reason

    # ── 6. Action-specific generation ────────────────────────
    if action == "ANSWER":
        response = generate_answer(query, compressed)
    elif action == "ASK":
        response = generate_clarification(query, reason=gate_reason)
    else:  # ABSTAIN
        response = generate_abstain(query, reason=gate_reason)

    trace["final_action"]   = action
    trace["final_response"] = response

    return {
        "query"            : query,
        "action"           : action,
        "response"         : response,
        "retrieved_chunks" : compressed,
        "num_retrieved"    : len(compressed),
        "trace"            : trace,
    }


# Quick test
print("="*55)
test_q = "Can I file for bankruptcy under Chapter 9?"
out    = decision_aware_rag(test_q)
print(f"Query    : {out['query']}")
print(f"Action   : {out['action']}")
print(f"Response : {out['response']}")
print(f"\nKey trace fields:")
print(f"  Gate reason    : {out['trace']['gate_reason']}")
print(f"  Classifier     : {out['trace']['classifier_label']}")
print(f"  Confidence     : {out['trace']['evidence_scores']['confidence_score']}")
print(f"  Coverage       : {out['trace']['evidence_scores']['coverage_score']}")
print(f"  Ambiguity      : {out['trace']['ambiguity_score']}")

In [ ]:
# ================================================================
# SECTION 6: EVALUATION — v3 vs v2 vs baseline
# ================================================================

from tqdm import tqdm

def sample_eval_set_v3(all_samples, n_per_action=150):
    """
    Increase eval size — 150 per action = 450 total
    Still balanced across ANSWER/ASK/ABSTAIN
    Also stratified across sources
    """
    random.seed(42)
    by_action = defaultdict(list)
    for s in all_samples:
        by_action[s["action"]].append(s)

    eval_set = []
    for action, samples in by_action.items():
        eval_set.extend(
            random.sample(samples, min(n_per_action, len(samples)))
        )
    random.shuffle(eval_set)
    return eval_set


def evaluate_v3(eval_samples, top_k_retrieve=20, top_k_rerank=5):
    results  = []
    correct  = 0
    hallucin = 0
    ask_is_question = 0   # NEW: track if ASK output is actually a question

    for sample in tqdm(eval_samples, desc="v3 Eval"):
        try:
            out = decision_aware_rag(
                sample["query"],
                top_k_retrieve = top_k_retrieve,
                top_k_rerank   = top_k_rerank,
                use_rewrite    = True,
                use_decompose  = True
            )
        except Exception as ex:
            print(f"Error: {ex}")
            out = {
                "action"           : "ABSTAIN",
                "response"         : "error",
                "retrieved_chunks" : [],
                "trace"            : {}
            }

        gt          = sample["action"]
        pred        = out["action"]
        is_correct  = (pred == gt)

        if is_correct:         correct  += 1
        if pred == "ANSWER" \
           and gt != "ANSWER": hallucin += 1
        if pred == "ASK" \
           and out["response"].strip().endswith("?"):
            ask_is_question += 1

        results.append({
            "sample_id"         : sample["id"],
            "source"            : sample["metadata"]["source"],
            "query"             : sample["query"],
            "gt_action"         : gt,
            "gt_response"       : sample["response"],
            "gt_failure_mode"   : sample["state"]["failure_mode"],
            "gt_difficulty"     : sample["state"]["difficulty"],
            "pred_action"       : pred,
            "pred_response"     : out["response"],
            "correct"           : is_correct,
            "gate_reason"       : out["trace"].get("gate_reason",""),
            "classifier_label"  : out["trace"].get("classifier_label",""),
            "confidence_score"  : out["trace"].get(
                "evidence_scores",{}).get("confidence_score",""),
            "ambiguity_score"   : out["trace"].get("ambiguity_score",""),
            "conflict_score"    : out["trace"].get("conflict_score",""),
            "top_chunks"        : [
                {
                    "source"      : c["source"],
                    "granularity" : c.get("granularity","—"),
                    "score"       : c.get("rerank_score",
                                    c.get("fused_score","—")),
                    "text_preview": c["text"][:120]
                }
                for c in out.get("retrieved_chunks",[])
            ]
        })

    n = len(results)
    n_ask_pred = sum(1 for r in results if r["pred_action"]=="ASK")

    print("\n" + "="*60)
    print("  DECISION-AWARE RAG v3 — EVALUATION RESULTS")
    print("="*60)
    print(f"  Total evaluated         : {n}")
    print(f"  Decision Accuracy       : {correct/n*100:.1f}%  ({correct}/{n})")
    print(f"  Hallucination Rate      : {hallucin/n*100:.1f}%")
    print(f"  ASK outputs that are ?  : {ask_is_question}/"
          f"{n_ask_pred}  "
          f"({'—' if n_ask_pred==0 else f'{100*ask_is_question/n_ask_pred:.1f}%'})")

    print(f"\n  --- Per-action accuracy ---")
    for act in ["ANSWER","ASK","ABSTAIN"]:
        act_res  = [r for r in results if r["gt_action"]==act]
        if not act_res: continue
        act_corr = sum(r["correct"] for r in act_res)
        print(f"  {act:10}: {act_corr}/{len(act_res)} "
              f"({100*act_corr/len(act_res):.1f}%)")

    print(f"\n  --- Confusion (GT → Predicted) ---")
    confusion = defaultdict(Counter)
    for r in results:
        confusion[r["gt_action"]][r["pred_action"]] += 1
    for gt, preds in sorted(confusion.items()):
        print(f"  GT={gt:10} → {dict(preds)}")

    print(f"\n  --- Accuracy by source ---")
    src_res = defaultdict(list)
    for r in results:
        src_res[r["source"]].append(r["correct"])
    for src, vals in sorted(src_res.items()):
        print(f"  {src:15}: {sum(vals)}/{len(vals)} "
              f"({100*sum(vals)/len(vals):.1f}%)")

    print(f"\n  --- Gate decisions breakdown ---")
    gate_counts = Counter(r.get("gate_reason","")[:40]
                          for r in results)
    for reason, cnt in gate_counts.most_common(10):
        print(f"  {cnt:4}x  {reason}")

    print(f"\n  --- Classifier label distribution ---")
    clf_counts = Counter(r["classifier_label"] for r in results)
    for label, cnt in clf_counts.items():
        print(f"  {label:25}: {cnt} ({100*cnt/n:.1f}%)")

    # Save
    v3_path = "/content/rag_pipeline/v3_eval_results.json"
    with open(v3_path, "w") as f:
        json.dump({
            "metrics": {
                "decision_accuracy"  : round(correct/n, 4),
                "hallucination_rate" : round(hallucin/n, 4),
                "ask_question_rate"  : round(ask_is_question/max(n_ask_pred,1),4),
                "n_evaluated"        : n,
            },
            "results": results
        }, f, indent=2)
    print(f"\nSaved → {v3_path}")
    return results


def compare_all_three():
    paths = {
        "Baseline RAG"   : "/content/rag_pipeline/eval_results.json",
        "Enhanced RAG"   : "/content/rag_pipeline/enhanced_eval_results.json",
        "Decision RAG v3": "/content/rag_pipeline/v3_eval_results.json",
    }

    # Filter to only existing files
    available = {name: path for name, path in paths.items()
                 if os.path.exists(path)}

    if not available:
        print("No eval result files found.")
        return

    if len(available) == 1:
        name, path = list(available.items())[0]
        print(f"\nOnly one eval file found: {name}")
        with open(path) as f:
            d = json.load(f)
        print(f"  Decision Accuracy  : {d['metrics']['decision_accuracy']*100:.1f}%")
        print(f"  Hallucination Rate : {d['metrics']['hallucination_rate']*100:.1f}%")
        print(f"  N evaluated        : {d['metrics']['n_evaluated']}")
        return

    # Multi-file comparison — only show available columns
    col_w = 13
    print("\n" + "="*70)
    print("  COMPARISON TABLE")
    print("="*70)
    header = f"  {'Metric':30}"
    for name in available:
        header += f" {name[:col_w]:>{col_w}}"
    print(header)
    print(f"  {'-'*68}")

    for metric in ["decision_accuracy", "hallucination_rate"]:
        row = f"  {metric:30}"
        for name, path in available.items():
            with open(path) as f:
                d = json.load(f)
            v    = d["metrics"].get(metric, 0)
            row += f" {v*100:>{col_w}.1f}%"
        print(row)

    print(f"\n  --- Per-action accuracy ---")
    for act in ["ANSWER", "ASK", "ABSTAIN"]:
        row = f"  {act:30}"
        for name, path in available.items():
            with open(path) as f:
                d = json.load(f)
            act_res  = [r for r in d["results"] if r["gt_action"] == act]
            if act_res:
                acc   = sum(r["correct"] for r in act_res) / len(act_res)
                row  += f" {acc*100:>{col_w}.1f}%"
            else:
                row  += f" {'—':>{col_w}}"
        print(row)

    # Per-action accuracy
    print(f"\n  --- Per-action accuracy ---")
    for act in ["ANSWER","ASK","ABSTAIN"]:
        row = f"  {act:30}"
        for name, path in paths.items():
            if os.path.exists(path):
                with open(path) as f:
                    d = json.load(f)
                act_res  = [r for r in d["results"]
                            if r["gt_action"]==act]
                if act_res:
                    acc = sum(r["correct"] for r in act_res)/len(act_res)
                    row += f" {acc*100:>11.1f}%"
                else:
                    row += f" {'—':>12}"
        print(row)


eval_samples_v3 = sample_eval_set_v3(all_samples, n_per_action=150)
print(f"Eval set v3: {len(eval_samples_v3)} samples | "
      f"{Counter(s['action'] for s in eval_samples_v3)}")

v3_results = evaluate_v3(eval_samples_v3)
compare_all_three()

In [ ]:
# ================================================================
# SECTION 7: INTERACTIVE DEBUG WINDOW — v3
# ================================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

query_input = widgets.Textarea(
    placeholder="Type your query here...",
    layout=widgets.Layout(width="700px", height="70px")
)
top_k_slider   = widgets.IntSlider(
    value=15, min=5, max=50, step=5,
    description="Retrieve K:",
    style={"description_width":"100px"},
    layout=widgets.Layout(width="380px")
)
rerank_slider  = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="Rerank to:",
    style={"description_width":"100px"},
    layout=widgets.Layout(width="380px")
)
alpha_slider   = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.1,
    description="BM25↔Dense α:",
    style={"description_width":"120px"},
    layout=widgets.Layout(width="380px"),
    readout_format=".1f"
)
tau_conf = widgets.FloatSlider(
    value=TAU_CONFIDENCE, min=0.0, max=1.0, step=0.05,
    description="τ confidence:",
    style={"description_width":"120px"},
    layout=widgets.Layout(width="380px"),
    readout_format=".2f"
)
tau_amb  = widgets.FloatSlider(
    value=TAU_AMBIGUITY, min=0.0, max=1.0, step=0.05,
    description="τ ambiguity:",
    style={"description_width":"120px"},
    layout=widgets.Layout(width="380px"),
    readout_format=".2f"
)
rewrite_toggle   = widgets.Checkbox(value=True, description="Query Rewrite")
decompose_toggle = widgets.Checkbox(value=True, description="Decompose")
run_button = widgets.Button(
    description  = "▶ Run v3 RAG",
    button_style = "primary",
    layout       = widgets.Layout(width="160px", height="40px")
)
out = widgets.Output()

ACTION_COLOR = {
    "ANSWER" : "#00e676",
    "ASK"    : "#40c4ff",
    "ABSTAIN": "#ff5252"
}

def render_v3_debug(result):
    trace   = result["trace"]
    action  = result["action"]
    color   = ACTION_COLOR.get(action, "#fff")
    chunks  = result.get("retrieved_chunks", [])
    ev      = trace.get("evidence_scores", {})

    # Signal gauges
    def gauge(label, val, good_high=True):
        if not isinstance(val, (int, float)):
            return ""
        pct    = int(val * 100)
        bar_col = "#00e676" if (good_high and val > 0.5) \
                  or (not good_high and val < 0.5) \
                  else "#ff5252"
        return f"""
        <div style="margin-bottom:4px;">
          <span style="color:#aaa; font-size:11px; width:140px;
                       display:inline-block;">{label}</span>
          <div style="display:inline-block; width:150px; height:10px;
                      background:#333; border-radius:4px; vertical-align:middle;">
            <div style="width:{pct}%; height:100%; background:{bar_col};
                        border-radius:4px;"></div>
          </div>
          <span style="color:#fff; font-size:11px; margin-left:6px;">
            {val:.3f}
          </span>
        </div>"""

    signals_html = (
        gauge("Confidence",  ev.get("confidence_score",0), good_high=True)  +
        gauge("Coverage",    ev.get("coverage_score",0),   good_high=True)  +
        gauge("Ambiguity",   trace.get("ambiguity_score",0), good_high=False)+
        gauge("Conflict",    trace.get("conflict_score",0),  good_high=False)
    )

    # Classifier badge
    clf_label = trace.get("classifier_label","—")
    clf_color = {"ANSWERABLE"           : "#00e676",
                 "NEEDS_CLARIFICATION"  : "#40c4ff",
                 "NOT_ANSWERABLE"       : "#ff5252"}.get(clf_label,"#aaa")

    # Chunks table
    chunk_rows = ""
    for i, c in enumerate(chunks, 1):
        score    = c.get("rerank_score", c.get("fused_score","—"))
        gran_col = "#80cbc4" if c.get("granularity")=="coarse" else "#ce93d8"
        chunk_rows += f"""
        <tr style="border-bottom:1px solid #2a2a2a;">
          <td style="padding:5px; color:#aaa;">{i}</td>
          <td style="padding:5px;">
            <span style="color:{gran_col}; font-size:10px;">
              [{c.get('granularity','—')}]
            </span>
            <span style="color:#80deea;"> {c['source']}</span>
          </td>
          <td style="padding:5px; color:#ffcc80;">{score}</td>
          <td style="padding:5px; color:#cfd8dc; font-size:11px;
                     white-space:pre-wrap;">{c['text'][:250]}...</td>
        </tr>"""

    html = f"""
    <div style="font-family:monospace; font-size:13px; background:#1e1e1e;
                color:#e0e0e0; padding:16px; border-radius:10px;
                border:1px solid #444;">

      <b style="color:#90caf9; font-size:14px;">
        🧠 Decision-Aware RAG v3 — Debug View
      </b>
      <hr style="border:0.5px solid #444; margin:8px 0;">

      <!-- Query -->
      <div style="margin-bottom:8px;">
        <b style="color:#90caf9;">Original Query:</b>
        <span style="color:#fff176;"> {result['query']}</span>
      </div>
      {"<div style='margin-bottom:8px;'><b style='color:#90caf9;'>Rewritten:</b> <span style='color:#ffcc80;'>" + trace.get('rewritten_q','—') + "</span></div>" if 'rewritten_q' in trace else ""}
      {"<div style='margin-bottom:8px;'><b style='color:#90caf9;'>Sub-queries:</b><br>" + "<br>".join(f"&nbsp;&nbsp;• {q}" for q in trace.get('sub_queries',[])) + "</div>" if 'sub_queries' in trace else ""}

      <hr style="border:0.5px solid #444; margin:8px 0;">

      <!-- Signals -->
      <div style="display:flex; gap:30px; margin-bottom:10px;">
        <div>
          <b style="color:#ce93d8;">📊 Evidence Signals</b><br><br>
          {signals_html}
        </div>
        <div>
          <b style="color:#ce93d8;">🤖 Classifier</b><br><br>
          <span style="color:{clf_color}; font-weight:bold;">
            {clf_label}
          </span><br>
          <span style="color:#aaa; font-size:11px;">
            {trace.get('classifier_reason','')[:80]}
          </span><br><br>
          <b style="color:#ce93d8;">⚡ Gate Decision</b><br>
          <span style="color:{color}; font-weight:bold;">
            {trace.get('gate_decision','—')}
          </span><br>
          <span style="color:#aaa; font-size:11px;">
            {trace.get('gate_reason','')[:100]}
          </span>
        </div>
      </div>
      <hr style="border:0.5px solid #444; margin:8px 0;">

      <!-- Chunks -->
      <b style="color:#90caf9;">📄 Retrieved Chunks
        ({trace.get('num_retrieved','—')} → reranked
        {trace.get('num_reranked','—')})
      </b><br>
      <table style="width:100%; border-collapse:collapse;
                    font-size:12px; margin-top:6px; margin-bottom:10px;">
        <tr style="color:#90caf9;">
          <th style="padding:4px; text-align:left;">#</th>
          <th style="padding:4px; text-align:left;">Gran | Source</th>
          <th style="padding:4px; text-align:left;">Score</th>
          <th style="padding:4px; text-align:left;">Text</th>
        </tr>
        {chunk_rows}
      </table>
      <hr style="border:0.5px solid #444; margin:8px 0;">

      <!-- Final output -->
      <div style="margin-bottom:6px;">
        <b style="color:#90caf9;">🎯 Final Action:</b>
        <span style="color:{color}; font-weight:bold; font-size:15px;">
          ● {action}
        </span>
      </div>
      <div style="background:#1b3a2b; padding:10px; border-radius:6px;
                  border-left:4px solid {color}; color:#c8e6c9;
                  font-size:13px;">
        {result['response']}
      </div>

    </div>
    """
    return html


def on_run_v3(b):
    query = query_input.value.strip()
    if not query:
        with out:
            clear_output()
            print("Please enter a query.")
        return

    # Update thresholds from sliders
    global TAU_CONFIDENCE, TAU_AMBIGUITY
    TAU_CONFIDENCE = tau_conf.value
    TAU_AMBIGUITY  = tau_amb.value

    with out:
        clear_output(wait=True)
        print("⏳ Running Decision-Aware RAG v3...")

    try:
        result = decision_aware_rag(
            query,
            top_k_retrieve = top_k_slider.value,
            top_k_rerank   = rerank_slider.value,
            alpha          = alpha_slider.value,
            use_rewrite    = rewrite_toggle.value,
            use_decompose  = decompose_toggle.value,
        )
        with out:
            clear_output(wait=True)
            display(HTML(render_v3_debug(result)))
    except Exception as ex:
        with out:
            clear_output(wait=True)
            print(f"❌ Error: {ex}")
            import traceback
            traceback.print_exc()


run_button.on_click(on_run_v3)

display(widgets.VBox([
    widgets.HTML(
        "<b style='color:#90caf9; font-size:14px;'>"
        "🧠 Decision-Aware RAG v3 — Interactive Debugger</b>"
    ),
    query_input,
    widgets.HBox([top_k_slider,   rerank_slider]),
    widgets.HBox([alpha_slider]),
    widgets.HBox([tau_conf,       tau_amb]),
    widgets.HTML(
        "<span style='color:#aaa; font-size:11px;'>"
        "τ confidence: below → ABSTAIN | "
        "τ ambiguity: above → ASK</span>"
    ),
    widgets.HBox([rewrite_toggle, decompose_toggle]),
    run_button,
    out
]))

In [ ]:
# ================================================================
# COMPREHENSIVE METRICS + PLOTS
# Works on whichever eval files exist
# ================================================================

!pip install rouge-score scikit-learn matplotlib seaborn -q

import json, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    accuracy_score,
    classification_report
)
from rouge_score import rouge_scorer
from collections import defaultdict, Counter

# ── Config ────────────────────────────────────────────────────
PATHS = {
    "Baseline"  : "/content/rag_pipeline/eval_results.json",
    "Enhanced"  : "/content/rag_pipeline/enhanced_eval_results.json",
    "v3 Decision": "/content/rag_pipeline/v3_eval_results.json",
}
ACTIONS  = ["ANSWER", "ASK", "ABSTAIN"]
COLORS   = {
    "Baseline"   : "#80cbc4",
    "Enhanced"   : "#80deea",
    "v3 Decision": "#ce93d8"
}
ACT_COL  = {"ANSWER":"#00e676", "ASK":"#40c4ff", "ABSTAIN":"#ff5252"}

os.makedirs("/content/rag_pipeline/plots", exist_ok=True)


# ================================================================
# LOAD ALL AVAILABLE FILES
# ================================================================

all_data = {}
for name, path in PATHS.items():
    if os.path.exists(path):
        with open(path) as f:
            all_data[name] = json.load(f)
        print(f"✅ Loaded: {name}  ({len(all_data[name]['results'])} samples)")
    else:
        print(f"⚠️  Not found: {name} — skipping")

if not all_data:
    raise FileNotFoundError("No eval result files found.")

names = list(all_data.keys())


# ================================================================
# HELPER FUNCTIONS
# ================================================================

def get_arrays(results):
    gt   = [r["gt_action"]   for r in results]
    pred = [r["pred_action"] for r in results]
    return gt, pred

def token_f1(pred_str, gt_str):
    """Token-level F1 between predicted and ground truth answer."""
    pred_toks = set(pred_str.lower().split())
    gt_toks   = set(gt_str.lower().split())
    if not pred_toks or not gt_toks:
        return 0.0
    common    = pred_toks & gt_toks
    if not common:
        return 0.0
    p = len(common) / len(pred_toks)
    r = len(common) / len(gt_toks)
    return 2 * p * r / (p + r)

def exact_match(pred_str, gt_str):
    return float(pred_str.strip().lower() == gt_str.strip().lower())

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def rouge_l(pred_str, gt_str):
    score = rouge.score(gt_str, pred_str)
    return score["rougeL"].fmeasure


# ================================================================
# 1. DECISION-LEVEL METRICS
# ================================================================

print("\n" + "="*70)
print("  1. DECISION-LEVEL METRICS")
print("="*70)

metrics_store = {}   # name → dict of all metrics

for name, data in all_data.items():
    results  = data["results"]
    gt, pred = get_arrays(results)
    n        = len(results)

    # Basic
    accuracy = accuracy_score(gt, pred)

    # Per-class P/R/F1
    prec, rec, f1, support = precision_recall_fscore_support(
        gt, pred, labels=ACTIONS, zero_division=0
    )
    macro_f1    = np.mean(f1)
    weighted_f1 = np.average(f1, weights=support)

    # Error behavior
    over_answer  = sum(1 for g,p in zip(gt,pred)
                       if p=="ANSWER" and g!="ANSWER") / n
    under_answer = sum(1 for g,p in zip(gt,pred)
                       if p=="ABSTAIN" and g=="ANSWER") / n

    # Misclassification matrix (raw counts)
    mis = defaultdict(lambda: defaultdict(int))
    for g, p in zip(gt, pred):
        if g != p:
            mis[g][p] += 1

    # Selective QA accuracy (only on predicted ANSWER)
    ans_pred  = [(g,p) for g,p in zip(gt,pred) if p=="ANSWER"]
    sel_acc   = sum(1 for g,p in ans_pred if g==p) / max(len(ans_pred),1)

    # Coverage
    coverage  = sum(1 for p in pred if p=="ANSWER") / n

    # Effective accuracy
    eff_acc   = accuracy   # same in this case but named separately

    # Confidence gap (if confidence_score exists)
    conf_correct = [r.get("confidence_score", None) for r in results
                    if r.get("confidence_score") and r["correct"]]
    conf_wrong   = [r.get("confidence_score", None) for r in results
                    if r.get("confidence_score") and not r["correct"]]
    conf_gap     = (np.mean(conf_correct) - np.mean(conf_wrong)
                    if conf_correct and conf_wrong else None)

    metrics_store[name] = {
        "accuracy"     : accuracy,
        "macro_f1"     : macro_f1,
        "weighted_f1"  : weighted_f1,
        "over_answer"  : over_answer,
        "under_answer" : under_answer,
        "selective_acc": sel_acc,
        "coverage"     : coverage,
        "conf_gap"     : conf_gap,
        "per_class"    : {
            act: {"precision": float(prec[i]),
                  "recall"   : float(rec[i]),
                  "f1"       : float(f1[i]),
                  "support"  : int(support[i])}
            for i, act in enumerate(ACTIONS)
        },
        "misclassification": {g: dict(v) for g, v in mis.items()},
        "n": n
    }

    print(f"\n{'─'*55}")
    print(f"  {name.upper()}")
    print(f"{'─'*55}")
    print(f"  Accuracy          : {accuracy*100:.1f}%")
    print(f"  Macro F1          : {macro_f1*100:.1f}%")
    print(f"  Weighted F1       : {weighted_f1*100:.1f}%")
    print(f"  Over-answer rate  : {over_answer*100:.1f}%  "
          f"(answered when shouldn't)")
    print(f"  Under-answer rate : {under_answer*100:.1f}%  "
          f"(abstained when should answer)")
    print(f"  Selective accuracy: {sel_acc*100:.1f}%  "
          f"(accuracy on predicted ANSWERs only)")
    print(f"  Coverage          : {coverage*100:.1f}%  "
          f"(% queries attempted)")
    if conf_gap:
        print(f"  Confidence gap    : {conf_gap:.4f}")

    print(f"\n  Per-class breakdown:")
    print(f"  {'Class':10} {'Prec':>8} {'Rec':>8} {'F1':>8} {'Support':>9}")
    for i, act in enumerate(ACTIONS):
        print(f"  {act:10} {prec[i]*100:>7.1f}% {rec[i]*100:>7.1f}% "
              f"{f1[i]*100:>7.1f}% {support[i]:>9}")

    print(f"\n  Misclassification counts:")
    for gt_act in ACTIONS:
        row = mis.get(gt_act, {})
        if row:
            for pred_act, cnt in row.items():
                print(f"  GT={gt_act:10} → Pred={pred_act:10}: {cnt:4} "
                      f"({100*cnt/n:.1f}%)")

    print(f"\n  Full classification report:")
    print(classification_report(gt, pred, labels=ACTIONS,
                                 zero_division=0))


# ================================================================
# 2. ANSWER QUALITY METRICS
# ================================================================

print("\n" + "="*70)
print("  2. ANSWER QUALITY METRICS (only on predicted ANSWER)")
print("="*70)

answer_quality = {}

for name, data in all_data.items():
    results = data["results"]

    # Only evaluate where GT=ANSWER and pred=ANSWER
    answer_results = [r for r in results
                      if r["gt_action"] == "ANSWER"
                      and r["pred_action"] == "ANSWER"]

    if not answer_results:
        print(f"  {name}: No correct ANSWER predictions found.")
        continue

    ems, tok_f1s, rouge_ls = [], [], []

    for r in answer_results:
        gt_resp   = r.get("gt_response", "")
        pred_resp = r.get("pred_response", r.get("response", ""))
        if not gt_resp or not pred_resp:
            continue
        ems.append(exact_match(pred_resp, gt_resp))
        tok_f1s.append(token_f1(pred_resp, gt_resp))
        rouge_ls.append(rouge_l(pred_resp, gt_resp))

    answer_quality[name] = {
        "n_answer_pairs": len(answer_results),
        "exact_match"   : np.mean(ems)    if ems    else 0.0,
        "token_f1"      : np.mean(tok_f1s) if tok_f1s else 0.0,
        "rouge_l"       : np.mean(rouge_ls) if rouge_ls else 0.0,
    }

    print(f"\n  {name}  (n={len(answer_results)} pairs)")
    print(f"  Exact Match  : {answer_quality[name]['exact_match']*100:.1f}%")
    print(f"  Token F1     : {answer_quality[name]['token_f1']*100:.1f}%")
    print(f"  ROUGE-L      : {answer_quality[name]['rouge_l']*100:.1f}%")


# ================================================================
# 3. COMBINED + COVERAGE METRICS
# ================================================================

print("\n" + "="*70)
print("  3. COMBINED METRICS")
print("="*70)

print(f"\n  {'Metric':30}", end="")
for name in names:
    print(f" {name[:14]:>14}", end="")
print()
print(f"  {'-'*70}")

combined_rows = [
    ("Effective Accuracy",  "accuracy"),
    ("Macro F1",            "macro_f1"),
    ("Weighted F1",         "weighted_f1"),
    ("Over-answer Rate",    "over_answer"),
    ("Under-answer Rate",   "under_answer"),
    ("Selective Accuracy",  "selective_acc"),
    ("Coverage",            "coverage"),
]

for label, key in combined_rows:
    print(f"  {label:30}", end="")
    for name in names:
        v = metrics_store[name].get(key, None)
        print(f" {v*100:>13.1f}%" if v is not None else f" {'—':>14}", end="")
    print()


# ================================================================
# 4. PLOTS
# ================================================================

print("\n  Generating plots...")

n_models  = len(names)
dark_bg   = "#1e1e1e"
panel_bg  = "#2a2a2a"

def style_ax(ax, title, xlabel="", ylabel=""):
    ax.set_facecolor(panel_bg)
    ax.title.set_color("white")
    ax.title.set_fontsize(10)
    ax.set_title(title)
    ax.tick_params(colors="#cccccc", labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#444")
    if xlabel: ax.set_xlabel(xlabel, color="#cccccc", fontsize=8)
    if ylabel: ax.set_ylabel(ylabel, color="#cccccc", fontsize=8)

fig = plt.figure(figsize=(22, 26))
fig.patch.set_facecolor(dark_bg)
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.5, wspace=0.38)


# ── Plot 1: Decision Accuracy by Model ───────────────────────
ax1  = fig.add_subplot(gs[0, 0])
accs = [metrics_store[n]["accuracy"]*100 for n in names]
bars = ax1.bar(names, accs,
               color=[COLORS[n] for n in names], edgecolor="#1e1e1e")
for bar, v in zip(bars, accs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f"{v:.1f}%", ha="center", color="white", fontsize=8)
style_ax(ax1, "Decision Accuracy", ylabel="%")
ax1.set_ylim(0, 100)
ax1.set_xticklabels(names, rotation=15)


# ── Plot 2: Macro F1 + Weighted F1 grouped ───────────────────
ax2   = fig.add_subplot(gs[0, 1])
x     = np.arange(n_models)
w     = 0.35
macro = [metrics_store[n]["macro_f1"]*100    for n in names]
wgtd  = [metrics_store[n]["weighted_f1"]*100 for n in names]
b1    = ax2.bar(x-w/2, macro, w, label="Macro F1",    color="#80cbc4")
b2    = ax2.bar(x+w/2, wgtd,  w, label="Weighted F1", color="#f48fb1")
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=15, fontsize=8)
ax2.legend(fontsize=7, facecolor="#333", labelcolor="white")
style_ax(ax2, "Macro vs Weighted F1", ylabel="%")
ax2.set_ylim(0, 100)


# ── Plot 3: Per-class F1 grouped bar ─────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
x   = np.arange(n_models)
w   = 0.25
for i, act in enumerate(ACTIONS):
    f1s = [metrics_store[n]["per_class"][act]["f1"]*100 for n in names]
    ax3.bar(x + (i-1)*w, f1s, w,
            label=act, color=ACT_COL[act], edgecolor="#1e1e1e")
ax3.set_xticks(x)
ax3.set_xticklabels(names, rotation=15, fontsize=8)
ax3.legend(fontsize=7, facecolor="#333", labelcolor="white")
style_ax(ax3, "Per-class F1 by Model", ylabel="F1 %")
ax3.set_ylim(0, 100)


# ── Plot 4-6: Confusion matrices ─────────────────────────────
for i, name in enumerate(names):
    ax  = fig.add_subplot(gs[1, i])
    results = all_data[name]["results"]
    gt, pred = get_arrays(results)
    cm  = confusion_matrix(gt, pred, labels=ACTIONS)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    im  = ax.imshow(cm_pct, cmap="YlOrRd", aspect="auto",
                    vmin=0, vmax=100)
    ax.set_xticks(range(len(ACTIONS)))
    ax.set_yticks(range(len(ACTIONS)))
    ax.set_xticklabels(ACTIONS, color="white", fontsize=8)
    ax.set_yticklabels(ACTIONS, color="white", fontsize=8)
    for r in range(len(ACTIONS)):
        for c in range(len(ACTIONS)):
            ax.text(c, r, f"{cm[r,c]}\n({cm_pct[r,c]:.0f}%)",
                    ha="center", va="center", fontsize=7,
                    color="black" if cm_pct[r,c]>50 else "white")
    ax.set_xlabel("Predicted", color="#cccccc", fontsize=8)
    ax.set_ylabel("Ground Truth", color="#cccccc", fontsize=8)
    style_ax(ax, f"Confusion — {name}")
    plt.colorbar(im, ax=ax, fraction=0.046).ax.yaxis.set_tick_params(
        color="white", labelcolor="white")


# ── Plot 7: Per-class Precision bar ──────────────────────────
ax7 = fig.add_subplot(gs[2, 0])
x   = np.arange(n_models)
w   = 0.25
for i, act in enumerate(ACTIONS):
    ps = [metrics_store[n]["per_class"][act]["precision"]*100
          for n in names]
    ax7.bar(x+(i-1)*w, ps, w, label=act,
            color=ACT_COL[act], edgecolor="#1e1e1e")
ax7.set_xticks(x)
ax7.set_xticklabels(names, rotation=15, fontsize=8)
ax7.legend(fontsize=7, facecolor="#333", labelcolor="white")
style_ax(ax7, "Per-class Precision", ylabel="%")
ax7.set_ylim(0, 100)


# ── Plot 8: Per-class Recall bar ──────────────────────────────
ax8 = fig.add_subplot(gs[2, 1])
for i, act in enumerate(ACTIONS):
    rs = [metrics_store[n]["per_class"][act]["recall"]*100
          for n in names]
    ax8.bar(x+(i-1)*w, rs, w, label=act,
            color=ACT_COL[act], edgecolor="#1e1e1e")
ax8.set_xticks(x)
ax8.set_xticklabels(names, rotation=15, fontsize=8)
ax8.legend(fontsize=7, facecolor="#333", labelcolor="white")
style_ax(ax8, "Per-class Recall", ylabel="%")
ax8.set_ylim(0, 100)


# ── Plot 9: Over/Under answer rate ───────────────────────────
ax9   = fig.add_subplot(gs[2, 2])
x     = np.arange(n_models)
w     = 0.35
over  = [metrics_store[n]["over_answer"]*100  for n in names]
under = [metrics_store[n]["under_answer"]*100 for n in names]
ax9.bar(x-w/2, over,  w, label="Over-answer",  color="#ff5252")
ax9.bar(x+w/2, under, w, label="Under-answer", color="#ffcc80")
ax9.set_xticks(x)
ax9.set_xticklabels(names, rotation=15, fontsize=8)
ax9.legend(fontsize=7, facecolor="#333", labelcolor="white")
style_ax(ax9, "Over/Under Answer Rate", ylabel="%")


# ── Plot 10: Coverage vs Selective Accuracy scatter ───────────
ax10 = fig.add_subplot(gs[3, 0])
for name in names:
    cov = metrics_store[name]["coverage"]*100
    sel = metrics_store[name]["selective_acc"]*100
    ax10.scatter(cov, sel, s=120,
                 color=COLORS[name], zorder=5, label=name)
    ax10.annotate(name, (cov, sel),
                  textcoords="offset points", xytext=(5,5),
                  color="white", fontsize=8)
style_ax(ax10, "Coverage vs Selective Accuracy",
         xlabel="Coverage %", ylabel="Selective Accuracy %")
ax10.legend(fontsize=7, facecolor="#333", labelcolor="white")


# ── Plot 11: Answer quality metrics (if available) ───────────
ax11 = fig.add_subplot(gs[3, 1])
if answer_quality:
    aq_names = list(answer_quality.keys())
    x        = np.arange(len(aq_names))
    w        = 0.25
    metrics_aq = ["exact_match", "token_f1", "rouge_l"]
    aq_colors  = ["#80cbc4", "#ce93d8", "#f48fb1"]
    for i, (m, c) in enumerate(zip(metrics_aq, aq_colors)):
        vals = [answer_quality[n][m]*100 for n in aq_names]
        ax11.bar(x+(i-1)*w, vals, w, label=m.replace("_"," ").title(),
                 color=c, edgecolor="#1e1e1e")
    ax11.set_xticks(x)
    ax11.set_xticklabels(aq_names, rotation=15, fontsize=8)
    ax11.legend(fontsize=7, facecolor="#333", labelcolor="white")
    style_ax(ax11, "Answer Quality (GT=ANSWER & Pred=ANSWER)",
             ylabel="%")
    ax11.set_ylim(0, 100)
else:
    ax11.text(0.5, 0.5, "No answer quality data",
              ha="center", va="center", color="white")
    style_ax(ax11, "Answer Quality")


# ── Plot 12: Radar / spider chart — per model ─────────────────
ax12 = fig.add_subplot(gs[3, 2], polar=True)
radar_metrics = ["accuracy","macro_f1","weighted_f1",
                 "selective_acc","coverage"]
radar_labels  = ["Accuracy","Macro F1","Weighted F1",
                 "Selective\nAcc","Coverage"]
N      = len(radar_metrics)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for name in names:
    vals = [metrics_store[name][m] for m in radar_metrics]
    vals += vals[:1]
    ax12.plot(angles, vals, linewidth=1.5,
              label=name, color=COLORS[name])
    ax12.fill(angles, vals, alpha=0.15, color=COLORS[name])

ax12.set_xticks(angles[:-1])
ax12.set_xticklabels(radar_labels, color="white", fontsize=8)
ax12.set_facecolor(panel_bg)
ax12.tick_params(colors="#cccccc")
ax12.set_title("Model Radar", color="white", fontsize=10, pad=15)
ax12.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1),
            fontsize=7, facecolor="#333", labelcolor="white")


plt.suptitle("RAG Comparative Study — Full Metrics Dashboard",
             color="white", fontsize=15, y=1.01)

plot_path = "/content/rag_pipeline/plots/full_metrics_dashboard.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight",
            facecolor=dark_bg)
plt.show()
print(f"\nSaved → {plot_path}")


# ================================================================
# 5. SAVE FULL METRICS SUMMARY JSON
# ================================================================

summary = {
    name: {
        "decision_metrics" : {
            k: v for k, v in metrics_store[name].items()
            if k != "misclassification"
        },
        "answer_quality"   : answer_quality.get(name, {}),
        "misclassification": metrics_store[name]["misclassification"]
    }
    for name in names
}

summary_path = "/content/rag_pipeline/metrics_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved metrics summary → {summary_path}")